# EDA - MUSEUMOGD.csv (Museen und Sammlungen)

City of Vienna open data export of museums/collections. Source file:
`data/raw/MUSEUMOGD.csv`. Run all cells to reproduce the findings below.

In [1]:
import pandas as pd
import re

df = pd.read_csv("../data/raw/MUSEUMOGD.csv")
df.shape

(137, 8)

## Columns & dtypes

In [2]:
df.dtypes

FID                     str
SHAPE                   str
NAME                    str
BEZIRK                int64
ADRESSE                 str
WEITERE_INF             str
SE_SDO_ROWID          int64
SE_ANNO_CAD_DATA    float64
dtype: object

## Missing values

In [3]:
df.isnull().sum()

FID                   0
SHAPE                 0
NAME                  0
BEZIRK                0
ADRESSE               0
WEITERE_INF           2
SE_SDO_ROWID          0
SE_ANNO_CAD_DATA    137
dtype: int64

## Duplicate check

Check for duplicate `FID` (should be unique feature IDs) and duplicate `NAME`.

In [4]:
print("Duplicate FID:", df["FID"].duplicated().sum())
print("Duplicate NAME:", df["NAME"].duplicated().sum())

Duplicate FID: 0
Duplicate NAME: 1


In [5]:
# Inspect the duplicate NAME rows directly
df[df.duplicated("NAME", keep=False)][["NAME", "BEZIRK", "ADRESSE"]]

,NAME,BEZIRK,ADRESSE
33,MAK - Museum für angewandte Kunst,1,"01., Stubenring 5"
125,MAK - Museum für angewandte Kunst,1,"01., Stubenring 5"


## Parse geometry

`SHAPE` is a WKT `POINT (lon lat)` string. Parse into separate `lon`/`lat` columns
and sanity-check the range against Vienna's bounding box.

In [6]:
def parse_point(s):
    m = re.match(r"POINT \(([\-0-9.]+) ([\-0-9.]+)\)", str(s))
    if m:
        return float(m.group(1)), float(m.group(2))
    return None, None

df["lon"], df["lat"] = zip(*df["SHAPE"].map(parse_point))

print("Unparseable SHAPE values:", df["lon"].isnull().sum())
print("lon range:", df["lon"].min(), "-", df["lon"].max())
print("lat range:", df["lat"].min(), "-", df["lat"].max())

Unparseable SHAPE values: 0
lon range: 16.256050906003573 - 16.508071606126386
lat range: 48.14510186654096 - 48.26372066627331


## District (`BEZIRK`) distribution

In [7]:
df["BEZIRK"].value_counts().sort_index()

BEZIRK
1     55
2      8
3      8
4      5
5      6
6      4
7      9
8      3
9      6
10     1
11     2
12     4
13     3
14     6
15     3
17     1
18     2
19     3
21     2
22     4
23     2
Name: count, dtype: int64

## `WEITERE_INF` (extra info / website link) coverage

In [8]:
print(df["WEITERE_INF"].notnull().sum(), "of", len(df), "rows have a value")
df["WEITERE_INF"].head(10)

135 of 137 rows have a value


0     http://www.chocolate-museum.wien
1         http://www.wienbibliothek.at
2              http://www.belvedere.at
3     http://dasrotewien-waschsalon.at
4    http://www.aspern-essling-1809.eu
5            https://www.westlicht.com
6          https://www.kindermuseum.at
7                                  NaN
8          http://www.bezirksmuseum.at
9                   http://www.khm.at/
Name: WEITERE_INF, dtype: str

## Sample addresses
Check the `ADRESSE` format - expected pattern: `"<district>., <Street> <number>"`.

In [9]:
df["ADRESSE"].head(10).tolist()

['05., Schönbrunnerstraße 99',
 '01., Bartensteingasse 9',
 '03., Rennweg 6',
 '19., Waschsalon Nr. 2, Halteraugasse 7',
 '22., Asperner Heldenplatz 9',
 '07., Westbahnstraße 40',
 '07., MuseumsQuartier, Museumsplatz 1',
 '01., Johannesgasse 6',
 '08., Schmiedgasse 18',
 '01., Hofburg, Neue Burg, Heldenplatz']

## Are the ArcGIS/SDE columns useful?

`SE_SDO_ROWID` and `SE_ANNO_CAD_DATA` look like internal export bookkeeping columns.

In [10]:
df[["SE_SDO_ROWID", "SE_ANNO_CAD_DATA"]].describe(include="all")

,SE_SDO_ROWID,SE_ANNO_CAD_DATA
count,137.000000,0.0
mean,138669.664234,NaN
std,110.217451,NaN
min,138412.000000,NaN
25%,138670.000000,NaN
50%,138704.000000,NaN
75%,138738.000000,NaN
max,138772.000000,NaN


## Random sample of cleaned fields

In [11]:
df[["NAME", "BEZIRK", "ADRESSE", "lon", "lat"]].sample(5, random_state=1)

,NAME,BEZIRK,ADRESSE,lon,lat
80,Wiener Aktionismus Museum,1,"01., Weihburggasse 26",16.375788,48.205172
5,WestLicht - Schauplatz für Fotografie,7,"07., Westbahnstraße 40",16.342122,48.201984
39,Kaffeemuseum Wien,5,"05., Vogelsanggasse 36",16.356711,48.183823
36,Wiener Ziegelmuseum,14,"14., Penzinger Straße 59",16.305082,48.189751
35,Bezirksmuseum Mariahilf,6,"06., Mollardgasse 8",16.352862,48.193675


## Findings

**Basics:** 137 records, 8 columns, comma-delimited.

**Geometry:** `SHAPE` parses cleanly for all 137 rows, already WGS84 - no
reprojection or geocoding needed. Coordinates fall within Vienna's bounds.

**Fields worth keeping:** `NAME`, `BEZIRK` (district, skews heavily to district 1 -
55/137, unsurprising given the Museumsquartier), `ADRESSE` (clean, consistent
`"<district>., <street> <number>"` format), `WEITERE_INF` (mostly a website URL,
135/137 populated).

**Fields to drop:** `SE_SDO_ROWID` and `SE_ANNO_CAD_DATA` are ArcGIS/SDE internal
bookkeeping columns - the latter is 100% null. `FID` is dataset-internal, fine to
drop once a KG-internal ID is minted.

**Data quality issues:** one exact duplicate row ("MAK - Museum für angewandte
Kunst", same district + address) - needs deduplication before ingestion. No missing
values otherwise.

**Suitability for the KG:** good candidate. Clean coordinates, consistent schema,
small size. Suggested minimal mapping: `NAME` → `poi:name`, `BEZIRK` →
`poi:district`, `lon`/`lat` → `geo:long`/`geo:lat`, `ADRESSE` → `poi:address`,
`WEITERE_INF` → `poi:website` (optional).